## Load Dataset

In [40]:
import pandas as pd
import numpy as np

In [18]:
train_df = pd.read_parquet("./train.parquet")
test_df = pd.read_parquet("./test.parquet")

## Data Cleaning

In [19]:
percentile_80 = train_df['ts_index'].quantile(0.80)
print(f"80th percentile of ts_index: {percentile_80}")

80th percentile of ts_index: 2967.0


In [20]:
train_df.columns

Index(['id', 'code', 'sub_code', 'sub_category', 'horizon', 'ts_index',
       'feature_a', 'feature_b', 'feature_c', 'feature_d', 'feature_e',
       'feature_f', 'feature_g', 'feature_h', 'feature_i', 'feature_j',
       'feature_k', 'feature_l', 'feature_m', 'feature_n', 'feature_o',
       'feature_p', 'feature_q', 'feature_r', 'feature_s', 'feature_t',
       'feature_u', 'feature_v', 'feature_w', 'feature_x', 'feature_y',
       'feature_z', 'feature_aa', 'feature_ab', 'feature_ac', 'feature_ad',
       'feature_ae', 'feature_af', 'feature_ag', 'feature_ah', 'feature_ai',
       'feature_aj', 'feature_ak', 'feature_al', 'feature_am', 'feature_an',
       'feature_ao', 'feature_ap', 'feature_aq', 'feature_ar', 'feature_as',
       'feature_at', 'feature_au', 'feature_av', 'feature_aw', 'feature_ax',
       'feature_ay', 'feature_az', 'feature_ba', 'feature_bb', 'feature_bc',
       'feature_bd', 'feature_be', 'feature_bf', 'feature_bg', 'feature_bh',
       'feature_bi', 'feature_

In [41]:
def weighted_rmse_score(y_target, y_pred, w):
    """Competition metric"""
    y_target = np.array(y_target, dtype=np.float64)
    y_pred = np.array(y_pred, dtype=np.float64)
    w = np.array(w, dtype=np.float64)
    
    denom = np.sum(w * y_target ** 2)
    if denom == 0 or np.isnan(denom):
        return 0.0
    
    numerator = np.sum(w * (y_target - y_pred) ** 2)
    ratio = numerator / denom
    clipped = np.clip(ratio, 0.0, 1.0)
    score = np.sqrt(1.0 - clipped)
    
    return float(score)

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import KFold
import pandas as pd


class TargetEncoder(BaseEstimator, TransformerMixin):
    """Simple, commented target encoder.

    - Encodes categorical columns by aggregating the target per category.
    - Supports multiple aggregations (e.g., 'mean', 'std').
    - `fit_transform` uses internal CV to avoid target leakage; smoothing
      for the 'mean' aggregation is applied per-fold only.

    Notes / caveats:
    - Smoothing is applied only for 'mean' inside `fit_transform`; `transform`
      uses the raw mappings learned in `fit` (no smoothing persisted).
    - Intended for numeric targets (regression) or binary targets encoded as
      0/1. Not directly suitable for multiclass targets without modification.
    - For time-series or grouped data, pass a suitable splitter (not supported
      by this class yet). KFold is used by default.
    """

    def __init__(self, cols_to_encode, aggs=['mean'], cv=5, smooth='auto', drop_original=False):
        # user parameters
        self.cols_to_encode = cols_to_encode
        self.aggs = aggs
        self.cv = cv
        self.smooth = smooth
        self.drop_original = drop_original

        # learned attributes populated in `fit`
        self.mappings_ = {}      # per-column, per-agg mapping series
        self.global_stats_ = {}  # global aggregated target for each agg

    def fit(self, X, y):
        """Learn mappings from the full dataset.

        These mappings are used by `transform` to encode new data. Note that no
        fold-based smoothing is applied here — mappings are the raw group aggs.
        """
        temp_df = X.copy()
        temp_df['target'] = y

        # global statistic for each aggregation (used for unseen categories)
        for agg_func in self.aggs:
            self.global_stats_[agg_func] = y.agg(agg_func)

        # store category -> aggregated target mappings for each column and agg
        for col in self.cols_to_encode:
            self.mappings_[col] = {}
            for agg_func in self.aggs:
                mapping = temp_df.groupby(col)['target'].agg(agg_func)
                self.mappings_[col][agg_func] = mapping

        return self

    def transform(self, X):
        """Apply learned mappings to `X`.

        - New columns are named `TE_<col>_<agg>`.
        - Categories not seen in `fit` are filled with the corresponding
          global statistic from `self.global_stats_`.
        - IMPORTANT: If you relied on fold-wise smoothing in `fit_transform`,
          `transform` will not apply that smoothing because `fit` stores raw
          group aggregates. Persisting smoothed values would require computing
          and storing them in `fit`.
        """
        X_transformed = X.copy()
        for col in self.cols_to_encode:
            for agg_func in self.aggs:
                new_col_name = f'TE_{col}_{agg_func}'
                map_series = self.mappings_[col][agg_func]
                X_transformed[new_col_name] = X[col].map(map_series)
                # fill unseen categories with global stat
                X_transformed[new_col_name].fillna(self.global_stats_[agg_func], inplace=True)

        if self.drop_original:
            X_transformed.drop(columns=self.cols_to_encode, inplace=True)

        return X_transformed

    def fit_transform(self, X, y):
        """Fit and transform using internal CV to prevent leakage.

        For each fold, mappings are learned on the training part and applied to
        the validation part. This prevents using the validation target to
        encode its own rows.

        Smoothing: only applied for the 'mean' aggregation. When `smooth='auto'`
        a simple empirical-Bayes heuristic is used (ratio of within-group and
        between-group variances). The smoothing is computed per-fold and used
        for that fold's encodings; these smoothed values are not written back
        to `self.mappings_` (so `transform` remains unsmoothed).
        """
        # fit on the whole data to populate mappings_ and global_stats_
        # (transform will use these mappings for new/unseen data)
        self.fit(X, y)

        # DataFrame to collect encoded features for all validation folds
        encoded_features = pd.DataFrame(index=X.index)

        kf = KFold(n_splits=self.cv, shuffle=True, random_state=42)

        for train_idx, val_idx in kf.split(X, y):
            X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
            X_val = X.iloc[val_idx]

            temp_df_train = X_train.copy()
            temp_df_train['target'] = y_train

            for col in self.cols_to_encode:
                # compute group-based statistics on fold's training data
                for agg_func in self.aggs:
                    new_col_name = f'TE_{col}_{agg_func}'

                    fold_global_stat = y_train.agg(agg_func)
                    mapping = temp_df_train.groupby(col)['target'].agg(agg_func)

                    if agg_func == 'mean':
                        # apply smoothing to the mean only
                        counts = temp_df_train.groupby(col)['target'].count()

                        m = self.smooth
                        if self.smooth == 'auto':
                            # empirical-Bayes style heuristic (may be unstable on tiny data)
                            variance_between = mapping.var()
                            avg_variance_within = temp_df_train.groupby(col)['target'].var().mean()
                            if variance_between > 0:
                                m = avg_variance_within / variance_between
                            else:
                                m = 0

                        # (counts * group_mean + m * global_mean) / (counts + m)
                        smoothed_mapping = (counts * mapping + m * fold_global_stat) / (counts + m)
                        encoded_values = X_val[col].map(smoothed_mapping)
                    else:
                        encoded_values = X_val[col].map(mapping)

                    # fill missing (unseen in training fold) with fold global stat
                    encoded_features.loc[X_val.index, new_col_name] = encoded_values.fillna(fold_global_stat)

        # attach encoded features to a copy of the original DataFrame
        X_transformed = X.copy()
        for col in encoded_features.columns:
            X_transformed[col] = encoded_features[col]

        if self.drop_original:
            X_transformed.drop(columns=self.cols_to_encode, inplace=True)

        return X_transformed

In [22]:
te = TargetEncoder(cols_to_encode=['code', 'sub_code', 'sub_category'],drop_original=True)
X_transformed = te.fit_transform(train_df, y=train_df['y_target'])

In [25]:
features = ['feature_a', 'feature_b', 'feature_c',
       'feature_d', 'feature_e', 'feature_f', 'feature_g', 'feature_h',
       'feature_i', 'feature_j', 'feature_k', 'feature_l', 'feature_m',
       'feature_n', 'feature_o', 'feature_p', 'feature_q', 'feature_r',
       'feature_s', 'feature_t', 'feature_u', 'feature_v', 'feature_w',
       'feature_x', 'feature_y', 'feature_z', 'feature_aa', 'feature_ab',
       'feature_ac', 'feature_ad', 'feature_ae', 'feature_af', 'feature_ag',
       'feature_ah', 'feature_ai', 'feature_aj', 'feature_ak', 'feature_al',
       'feature_am', 'feature_an', 'feature_ao', 'feature_ap', 'feature_aq',
       'feature_ar', 'feature_as', 'feature_at', 'feature_au', 'feature_av',
       'feature_aw', 'feature_ax', 'feature_ay', 'feature_az', 'feature_ba',
       'feature_bb', 'feature_bc', 'feature_bd', 'feature_be', 'feature_bf',
       'feature_bg', 'feature_bh', 'feature_bi', 'feature_bj', 'feature_bk',
       'feature_bl', 'feature_bm', 'feature_bn', 'feature_bo', 'feature_bp',
       'feature_bq', 'feature_br', 'feature_bs', 'feature_bt', 'feature_bu',
       'feature_bv', 'feature_bw', 'feature_bx', 'feature_by', 'feature_bz',
       'feature_ca', 'feature_cb', 'feature_cc', 'feature_cd', 'feature_ce',
       'feature_cf', 'feature_cg', 'feature_ch', 'TE_code_mean', 'TE_sub_code_mean', 'TE_sub_category_mean']

target = "y_target"

In [28]:
train_mask = X_transformed['ts_index'] <= percentile_80
val_mask = X_transformed['ts_index'] > percentile_80


In [ ]:
import lightgbm as lgb
import gc

models = {}
scores = {}

params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.02,
    'num_leaves': 128,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'seed': 42,
    'verbosity': -1,
    'n_jobs': -1
}

horizons = X_transformed['horizon'].unique()
for h in horizons:

    print(f"\n====================")
    print(f"Training horizon: {h}")
    print(f"====================")

    X_transformed_h = X_transformed[X_transformed['horizon']==h]
    percentile_80 = X_transformed_h['ts_index'].quantile(0.80)
    print(f"80th percentile of ts_index: {percentile_80}")
    # --- Split ---
    X_train = X_transformed_h[train_mask][features]
    y_train = X_transformed_h[train_mask][target]
    w_train = X_transformed_h[train_mask]['weight']

    X_val = X_transformed_h[val_mask][features]
    y_val = X_transformed_h[val_mask][target]
    w_val = X_transformed_h[val_mask]['weight']

    # --- LightGBM Dataset ---
    train_data = lgb.Dataset(X_train, label=y_train, weight=w_train)
    val_data = lgb.Dataset(X_val, label=y_val, weight=w_val, reference=train_data)

    # --- Train ---
    model = lgb.train(
        params,
        train_data,
        num_boost_round=5000,
        valid_sets=[train_data, val_data],
        valid_names=['train', 'valid'],
        callbacks=[
            lgb.early_stopping(stopping_rounds=50),
            lgb.log_evaluation(period=50)
        ]
    )

    print(f"Best iteration ({h}): {model.best_iteration}")
    print(f"Best validation RMSE ({h}): {model.best_score['valid']['rmse']}")

    # --- Validation prediction ---
    y_pred_val = model.predict(X_val, num_iteration=model.best_iteration)

    score = weighted_rmse_score(y_val, y_pred_val, w_val)
    print(f">>> Weighted RMSE ({h}): {score:.5f}")

    scores[h] = score
    models[h] = model

    # --- Memory cleanup ---
    del train_data, val_data
    gc.collect()



Training horizon: 25
80th percentile of ts_index: 2964.0


C:\Users\hp\AppData\Local\Temp\ipykernel_10084\1297493095.py:31: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  X_train = X_transformed_h[train_mask][features]
C:\Users\hp\AppData\Local\Temp\ipykernel_10084\1297493095.py:32: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  y_train = X_transformed_h[train_mask][target]
C:\Users\hp\AppData\Local\Temp\ipykernel_10084\1297493095.py:33: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  w_train = X_transformed_h[train_mask]['weight']
C:\Users\hp\AppData\Local\Temp\ipykernel_10084\1297493095.py:35: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  X_val = X_transformed_h[val_mask][features]
C:\Users\hp\AppData\Local\Temp\ipykernel_10084\1297493095.py:36: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  y_val = X_transformed_h[val_mask][target]
C:\Users\hp\AppData\Local\Temp\ipykernel_10084\1297493095

Training until validation scores don't improve for 50 rounds
[50]	train's rmse: 0.00337416	valid's rmse: 0.00412085
[100]	train's rmse: 0.00325809	valid's rmse: 0.00411758
Early stopping, best iteration is:
[95]	train's rmse: 0.00326825	valid's rmse: 0.00411727
Best iteration (25): 95
Best validation RMSE (25): 0.004117269285447959
>>> Weighted RMSE (25): 0.16768

Training horizon: 1
80th percentile of ts_index: 2969.0


C:\Users\hp\AppData\Local\Temp\ipykernel_10084\1297493095.py:31: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  X_train = X_transformed_h[train_mask][features]
C:\Users\hp\AppData\Local\Temp\ipykernel_10084\1297493095.py:32: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  y_train = X_transformed_h[train_mask][target]
C:\Users\hp\AppData\Local\Temp\ipykernel_10084\1297493095.py:33: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  w_train = X_transformed_h[train_mask]['weight']
C:\Users\hp\AppData\Local\Temp\ipykernel_10084\1297493095.py:35: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  X_val = X_transformed_h[val_mask][features]
C:\Users\hp\AppData\Local\Temp\ipykernel_10084\1297493095.py:36: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  y_val = X_transformed_h[val_mask][target]
C:\Users\hp\AppData\Local\Temp\ipykernel_10084\1297493095

Training until validation scores don't improve for 50 rounds
[50]	train's rmse: 0.000943025	valid's rmse: 0.00115913
[100]	train's rmse: 0.000933616	valid's rmse: 0.00115935
Early stopping, best iteration is:
[63]	train's rmse: 0.000940304	valid's rmse: 0.00115905
Best iteration (1): 63
Best validation RMSE (1): 0.0011590455718856055
>>> Weighted RMSE (1): 0.04287

Training horizon: 3
80th percentile of ts_index: 2968.0


C:\Users\hp\AppData\Local\Temp\ipykernel_10084\1297493095.py:31: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  X_train = X_transformed_h[train_mask][features]
C:\Users\hp\AppData\Local\Temp\ipykernel_10084\1297493095.py:32: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  y_train = X_transformed_h[train_mask][target]
C:\Users\hp\AppData\Local\Temp\ipykernel_10084\1297493095.py:33: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  w_train = X_transformed_h[train_mask]['weight']
C:\Users\hp\AppData\Local\Temp\ipykernel_10084\1297493095.py:35: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  X_val = X_transformed_h[val_mask][features]
C:\Users\hp\AppData\Local\Temp\ipykernel_10084\1297493095.py:36: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  y_val = X_transformed_h[val_mask][target]
C:\Users\hp\AppData\Local\Temp\ipykernel_10084\1297493095

Training until validation scores don't improve for 50 rounds
[50]	train's rmse: 0.00113497	valid's rmse: 0.00198916
Early stopping, best iteration is:
[43]	train's rmse: 0.00113729	valid's rmse: 0.00198096
Best iteration (3): 43
Best validation RMSE (3): 0.0019809619179024748
>>> Weighted RMSE (3): 0.06886

Training horizon: 10
80th percentile of ts_index: 2965.0


C:\Users\hp\AppData\Local\Temp\ipykernel_10084\1297493095.py:31: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  X_train = X_transformed_h[train_mask][features]
C:\Users\hp\AppData\Local\Temp\ipykernel_10084\1297493095.py:32: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  y_train = X_transformed_h[train_mask][target]
C:\Users\hp\AppData\Local\Temp\ipykernel_10084\1297493095.py:33: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  w_train = X_transformed_h[train_mask]['weight']
C:\Users\hp\AppData\Local\Temp\ipykernel_10084\1297493095.py:35: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  X_val = X_transformed_h[val_mask][features]
C:\Users\hp\AppData\Local\Temp\ipykernel_10084\1297493095.py:36: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  y_val = X_transformed_h[val_mask][target]
C:\Users\hp\AppData\Local\Temp\ipykernel_10084\1297493095

Training until validation scores don't improve for 50 rounds
[50]	train's rmse: 0.00256016	valid's rmse: 0.00319705
Early stopping, best iteration is:
[42]	train's rmse: 0.00257271	valid's rmse: 0.00319377
Best iteration (10): 42
Best validation RMSE (10): 0.00319377323802841
>>> Weighted RMSE (10): 0.08221


In [ ]:
X_test = te.transform(test_df)
test_preds = model.predict(X_test[features], num_iteration=model.best_iteration)

{np.int32(25): 0.16767853769507632,
 np.int32(1): 0.04286681162718284,
 np.int32(3): 0.06886288512569505,
 np.int32(10): 0.0822104919041432}

In [51]:
X_test = te.transform(test_df)


C:\Users\hp\AppData\Local\Temp\ipykernel_10084\1242350735.py:75: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  X_transformed[new_col_name].fillna(self.global_stats_[agg_func], inplace=True)
C:\Users\hp\AppData\Local\Temp\ipykernel_10084\1242350735.py:75: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always b

In [52]:
X_test

,id,horizon,ts_index,feature_a,feature_b,feature_c,feature_d,feature_e,feature_f,feature_g,...,feature_cb,feature_cc,feature_cd,feature_ce,feature_cf,feature_cg,feature_ch,TE_code_mean,TE_sub_code_mean,TE_sub_category_mean
0,W2MW3G2L__495MGHFJ__PZ9S1Z4V__3__3647,3,3647,95,10.365266,3.209321,8.109339,9.043471,10.123041,15.722121,...,-0.058961,-0.002774,-0.001480,-0.256460,1.665532,0.071324,2,-0.022722,-0.553854,-1.217609
1,W2MW3G2L__495MGHFJ__PZ9S1Z4V__10__3647,10,3647,88,2.571477,15.234848,16.505699,0.230426,10.145378,10.159641,...,-0.058961,-0.002774,-0.001480,-0.256460,1.665532,0.071324,2,-0.022722,-0.553854,-1.217609
2,W2MW3G2L__495MGHFJ__PZ9S1Z4V__25__3647,25,3647,71,5.524709,6.931663,8.939537,0.668187,16.578701,3.150690,...,-0.058961,-0.002774,-0.001480,-0.256460,1.665532,0.071324,2,-0.022722,-0.553854,-1.217609
3,W2MW3G2L__495MGHFJ__PZ9S1Z4V__1__3647,1,3647,97,10.293758,14.893660,9.435544,2.335377,3.477961,15.680595,...,-0.058961,-0.002774,-0.001480,-0.256460,1.665532,0.071324,2,-0.022722,-0.553854,-1.217609
4,W2MW3G2L__495MGHFJ__PZ9S1Z4V__10__3648,10,3648,87,14.776194,7.701180,6.228968,13.118262,2.762462,0.598215,...,-0.059835,-0.002838,-0.001501,-0.242240,1.671890,0.071100,2,-0.022722,-0.553854,-1.217609
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1447102,83EG83KQ__VYN97209__PHHHVYZI__3__4305,3,4305,1,16.314806,16.075016,4.102829,3.298536,0.672089,10.431427,...,-0.000326,-0.017148,-0.005026,0.167489,0.022252,0.072114,0,0.000018,-0.665905,-1.446897
1447103,83EG83KQ__VYN97209__PHHHVYZI__1__4306,1,4306,2,4.681250,7.401711,7.197559,8.557788,7.588660,14.246598,...,-0.000337,-0.017592,-0.005727,0.171962,0.021623,0.069115,0,0.000018,-0.665905,-1.446897
1447104,83EG83KQ__VYN97209__PHHHVYZI__3__4306,3,4306,0,5.372833,13.592936,16.349126,8.121506,11.596113,12.532656,...,-0.000337,-0.017592,-0.005727,0.171962,0.021623,0.069115,0,0.000018,-0.665905,-1.446897
1447105,83EG83KQ__VYN97209__PHHHVYZI__1__4307,1,4307,1,7.543404,3.597098,11.375947,7.543607,4.051237,0.679651,...,-0.000351,-0.019842,-0.006515,0.196026,0.021906,0.067739,0,0.000018,-0.665905,-1.446897


In [ ]:
X_test['prediction'] = np.nan
for h, model in models.items():
    print(f"Predicting horizon: {h}")

    mask = X_test['horizon'] == h

    X_test_h = X_test.loc[mask, features]

    X_test.loc[mask, 'prediction'] = model.predict(
        X_test_h,
        num_iteration=model.best_iteration
    )
